# Vehicle Route Optimization using Genetic Algorithm
### Soft Computing Techniques — Mini Project

**Objective:** Given a set of delivery/visit locations (each with a name and 2D coordinates),
find the shortest possible route that visits every location exactly once and returns to the
starting point (a classic **Traveling Salesman Problem**, solved here using a **Genetic
Algorithm (GA)**).

**Why Genetic Algorithm (Soft Computing)?**
- The search space of possible routes grows factorially with the number of locations `((n-1)!/2)`,
  making exact/brute-force search infeasible beyond ~10-12 locations.
- GA is a nature-inspired metaheuristic (based on Darwinian evolution: selection, crossover,
  mutation) that can find near-optimal solutions in a reasonable time without needing the
  problem's exact mathematical structure — a classic soft computing approach.

**Pipeline used in this notebook:**
1. Take location names + coordinates as input.
2. Represent a *route* as a chromosome (a permutation of location indices).
3. Evaluate *fitness* as the inverse of total route distance.
4. Evolve the population using **Tournament Selection**, **Order Crossover (OX)**,
   **Swap Mutation**, and **Elitism** across many generations.
5. Track convergence and visualize the final optimized route.

This same GA logic is reused (in `app.py`) to power the Streamlit front-end.

## 1. Import Libraries

In [ ]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)


## 2. Problem Setup — Define Locations

You can either:
- Set `USE_SAMPLE_DATA = True` to run instantly with a demo set of locations, **or**
- Set `USE_SAMPLE_DATA = False` to type in your own locations interactively (name + x + y coordinates)
  when running this cell in Colab.

Coordinates can represent anything consistent — km on a map, lat/long scaled, or plain grid units.

In [ ]:
USE_SAMPLE_DATA = True  # <-- change to False to enter your own locations interactively

if USE_SAMPLE_DATA:
    locations = {
        "A": (0, 0),
        "B": (2, 6),
        "C": (5, 2),
        "D": (7, 8),
        "E": (9, 1),
        "F": (3, 9),
        "G": (6, 5),
        "H": (1, 4),
    }
else:
    locations = {}
    n = int(input("Enter number of locations: "))
    for i in range(n):
        name = input(f"Enter name for location {i+1} (e.g. A, B, Warehouse): ").strip()
        x = float(input(f"  Enter X coordinate for {name}: "))
        y = float(input(f"  Enter Y coordinate for {name}: "))
        locations[name] = (x, y)

names = list(locations.keys())
coords = np.array([locations[n] for n in names])
print("Locations loaded:")
for n_, (x, y) in locations.items():
    print(f"  {n_}: ({x}, {y})")


## 3. Genetic Algorithm Components

**Chromosome representation:** a route is a permutation of location indices, e.g. `[0, 3, 1, 2]`
means the tour visits `A -> D -> B -> C -> A` (it always returns to the start).

**Fitness function:** `fitness = 1 / total_route_distance` — shorter routes get higher fitness.

In [ ]:
def total_distance(route, coords):
    """Total Euclidean distance of a closed tour (returns to start)."""
    dist = 0.0
    n = len(route)
    for i in range(n):
        a = coords[route[i]]
        b = coords[route[(i + 1) % n]]
        dist += math.dist(a, b)
    return dist


def fitness(route, coords):
    d = total_distance(route, coords)
    return 1.0 / d if d > 0 else float("inf")


def create_route(num_cities):
    route = list(range(num_cities))
    random.shuffle(route)
    return route


def initial_population(pop_size, num_cities):
    return [create_route(num_cities) for _ in range(pop_size)]


In [ ]:
def tournament_selection(population, fitnesses, k=5):
    """Pick k random individuals, return the fittest among them."""
    contenders = random.sample(range(len(population)), k)
    best = max(contenders, key=lambda idx: fitnesses[idx])
    return population[best]


In [ ]:
def ordered_crossover(parent1, parent2):
    """OX1: preserves a slice from parent1, fills the rest in parent2's order."""
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))

    child = [None] * size
    child[start:end] = parent1[start:end]

    fill_values = [g for g in parent2 if g not in child]
    pointer = 0
    for i in range(size):
        if child[i] is None:
            child[i] = fill_values[pointer]
            pointer += 1
    return child


In [ ]:
def swap_mutation(route, mutation_rate):
    """Each gene has `mutation_rate` chance of swapping position with another random gene."""
    route = route[:]
    for i in range(len(route)):
        if random.random() < mutation_rate:
            j = random.randint(0, len(route) - 1)
            route[i], route[j] = route[j], route[i]
    return route


## 4. The Genetic Algorithm Main Loop

Uses **elitism** (best individuals are carried over unchanged) combined with
tournament selection, ordered crossover, and swap mutation to evolve the population
across generations.

In [ ]:
def genetic_algorithm(coords, pop_size=150, generations=400, elite_size=20,
                       mutation_rate=0.02, tournament_k=5, verbose=True):
    num_cities = len(coords)
    population = initial_population(pop_size, num_cities)
    best_distance_history = []
    best_route_overall = None
    best_distance_overall = float("inf")

    for gen in range(generations):
        fitnesses = [fitness(r, coords) for r in population]

        ranked = sorted(zip(population, fitnesses), key=lambda x: x[1], reverse=True)
        population = [r for r, f in ranked]
        fitnesses = [f for r, f in ranked]

        current_best_route = population[0]
        current_best_distance = total_distance(current_best_route, coords)
        if current_best_distance < best_distance_overall:
            best_distance_overall = current_best_distance
            best_route_overall = current_best_route[:]
        best_distance_history.append(best_distance_overall)

        next_population = population[:elite_size]  # elitism
        while len(next_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses, tournament_k)
            parent2 = tournament_selection(population, fitnesses, tournament_k)
            child = ordered_crossover(parent1, parent2)
            child = swap_mutation(child, mutation_rate)
            next_population.append(child)

        population = next_population

        if verbose and (gen % 50 == 0 or gen == generations - 1):
            print(f"Generation {gen:4d}  |  Best distance so far: {best_distance_overall:.3f}")

    return best_route_overall, best_distance_overall, best_distance_history


## 5. Run the Genetic Algorithm

In [ ]:
best_route, best_distance, history = genetic_algorithm(
    coords,
    pop_size=150,
    generations=400,
    elite_size=20,
    mutation_rate=0.02,
    tournament_k=5,
)

route_names = [names[i] for i in best_route] + [names[best_route[0]]]
print("\nOptimal route found:")
print("  -> ".join(route_names))
print(f"\nTotal route distance: {best_distance:.3f} units")


## 6. Visualization

### 6.1 Convergence Curve — GA improving the solution over generations

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history, color="#2E86AB", linewidth=2)
plt.title("GA Convergence: Best Route Distance per Generation")
plt.xlabel("Generation")
plt.ylabel("Best Distance Found")
plt.grid(alpha=0.3)
plt.show()


### 6.2 Optimized Route Plot

In [ ]:
plt.figure(figsize=(7, 7))

tour_coords = coords[best_route + [best_route[0]]]
plt.plot(tour_coords[:, 0], tour_coords[:, 1], "o-", color="#E63946", linewidth=2, markersize=10, zorder=1)

for i, city_idx in enumerate(best_route):
    x, y = coords[city_idx]
    plt.scatter(x, y, s=180, color="#1D3557", zorder=2)
    plt.text(x + 0.15, y + 0.15, names[city_idx], fontsize=12, fontweight="bold")

plt.title(f"Optimized Vehicle Route (Total Distance: {best_distance:.2f})")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.grid(alpha=0.3)
plt.axis("equal")
plt.show()


## 7. Conclusion & Notes

- The Genetic Algorithm successfully evolved a population of random routes into a
  near-optimal tour, visibly minimizing total travel distance across generations
  (see the convergence curve).
- **Tunable hyperparameters:** population size, number of generations, mutation rate,
  elite size, and tournament size all affect solution quality vs. speed.
  Larger populations / more generations generally converge to better routes but take longer.
- **Extensions for future work:**
  - Multiple vehicles (splitting locations into `k` sub-routes) → *Vehicle Routing Problem (VRP)*.
  - Capacity constraints per vehicle.
  - Real road-network distances (via a routing API) instead of straight-line Euclidean distance.
  - Comparing GA against other soft-computing methods (Ant Colony Optimization, PSO, Simulated Annealing).

This same algorithm (functions above) is reused inside the Streamlit web app (`app.py`)
so users can interactively enter their own locations and see the optimized route and
convergence chart rendered live in the browser.